# Response streaming

Without streaming you wait for the whole answer. With streaming the text arrives in chunks
as it is generated, so a chat UI can show it right away.

- `client.messages.create(..., stream=True)` yields raw events
- `client.messages.stream(...)` is the SDK helper: `stream.text_stream` yields only text
- `stream.get_final_message()` returns the complete message after streaming

## Setup

The client reads `ANTHROPIC_API_KEY` (and `ANTHROPIC_BASE_URL`, if set) from `.env` in the repo root.

In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## Helpers

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

In [3]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

## 1. Raw events

`message_start`, then `content_block_start`, many `content_block_delta` events with text pieces,
`content_block_stop`, `message_delta` (stop reason and usage) and `message_stop`.

In [4]:
stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True,
)

for event in stream:
    print(event.type, getattr(event, "delta", ""))

message_start 
content_block_start 
content_block_delta TextDelta(text='#', type='text_delta')
content_block_delta TextDelta(text=' F', type='text_delta')
content_block_delta TextDelta(text='akeDB', type='text_delta')
content_block_delta TextDelta(text='\n\nF', type='text_delta')
content_block_delta TextDelta(text='akeDB is an', type='text_delta')
content_block_delta TextDelta(text=' in', type='text_delta')


content_block_delta TextDelta(text='-memory database system', type='text_delta')
content_block_delta TextDelta(text=' that', type='text_delta')
content_block_delta TextDelta(text=' generates', type='text_delta')
content_block_delta TextDelta(text=' realistic', type='text_delta')
content_block_delta TextDelta(text=' mock data across', type='text_delta')
content_block_delta TextDelta(text=' thousands', type='text_delta')
content_block_delta TextDelta(text=' of tables and', type='text_delta')
content_block_delta TextDelta(text=' automatically', type='text_delta')
content_block_delta TextDelta(text=' scales', type='text_delta')
content_block_delta TextDelta(text=' to', type='text_delta')
content_block_delta TextDelta(text=' handle sim', type='text_delta')
content_block_delta TextDelta(text='ulated enterprise', type='text_delta')
content_block_delta TextDelta(text=' work', type='text_delta')
content_block_delta TextDelta(text='loads without', type='text_delta')
content_block_delta TextDelta

## 2. Text only

`flush=True` makes the terminal show each chunk immediately.

In [5]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

#

 Fake Database Description



**

Mem

oryV

ault DB

... (17 outputs left out)

 feelings

 before

 they occur

.

## 3. Stream, then keep the complete message

In [6]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        # Send each chunk to your client
        pass

    # Get the complete message for database storage
    final_message = stream.get_final_message()

print("".join(block.text for block in final_message.content if block.type == "text"))
print("stop_reason:", final_message.stop_reason)
print("output tokens:", final_message.usage.output_tokens)

# FakeDB

FakeDB is an in-memory NoSQL database that generates realistic synthetic data with customizable schemas, perfect for rapid prototyping and testing without connecting to production systems.
stop_reason: end_turn
output tokens: 43
